# 01 - FSL FAST + MNI alignment of the 20 HCP healthy scans

Run on **Kaggle** with: Accelerator = None (CPU is enough), **Internet = On**, and the HCP files attached as a **private** Dataset.

Steps: (1) install FSL tools, (2) get our code, (3) test on **1 subject** and look at the picture, (4) run all 20, (5) download results.

## 1. Install FSL (FAST, FLIRT, MNI templates) with micromamba - takes a few minutes

In [ ]:
import os, subprocess
def sh(cmd):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout[-3000:], r.stderr[-3000:]); return r.returncode

sh("mkdir -p /tmp/mm && cd /tmp/mm && curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba")
rc = sh("/tmp/mm/bin/micromamba create -y -p /tmp/fsl-env -c https://fsl.fmrib.ox.ac.uk/fsldownloads/fslconda/public/ -c conda-forge fsl-fast4 fsl-flirt fsl-data_standard")
print("install return code:", rc)

os.environ["FSLDIR"] = "/tmp/fsl-env"
os.environ["FSLOUTPUTTYPE"] = "NIFTI_GZ"
os.environ["PATH"] = "/tmp/fsl-env/bin:" + os.environ["PATH"]
sh("which fast flirt; ls /tmp/fsl-env/data/standard | head -30")

**Check the output above:** `fast` and `flirt` must be found, and the list should include `MNI152_T1_1mm_brain.nii.gz`.
If the template is missing, stop and tell me - do not continue.

## 2. Get our code (private repo). Add a Kaggle Secret called `GITHUB_TOKEN` (a GitHub personal access token with read access), or upload the repo as a Dataset instead.

In [ ]:
import os, sys
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    sh(f"cd /kaggle/working && rm -rf repo && git clone https://{token}@github.com/PiyushGarg0146/lf-synthseg-btp.git repo")
except Exception as e:
    print("No token / clone failed:", e, "
Attach the repo as a Dataset and set REPO below.")
REPO = "/kaggle/working/repo"      # change if you uploaded the repo as a Dataset
sh(f"pip install -q nibabel scipy matplotlib pyyaml scikit-learn; ls {REPO}")

## 3. Test on ONE subject first

In [ ]:
HCP = "/kaggle/input"              # searched recursively for *.nii.gz
OUT = "/kaggle/working/hcp_processed"
sh(f"cd {REPO} && python scripts/fsl_pipeline.py --src {HCP} --dst {OUT} --limit 1")

In [ ]:
from IPython.display import Image, display
import glob
for p in glob.glob(OUT + "/*/qc.png"): display(Image(p))
print(open(OUT + "/summary.csv").read())

### Look at the picture and the table before going on
* Bottom row should show brain tissue in sensible colours: CSF (1) in the ventricles and around the brain, GM (2) as a thin ribbon at the cortex, WM (3) inside.
* The printed line `mean T2 CSF, GM, WM, dark` should be decreasing (CSF brightest). If a WARNING is printed, the label mapping in `data/labels.py` is wrong.
* Check what class 4 ("dark") looks like and how big it is. **This is the part of the paper that is unclear.** Send me the picture.

## 4. Run all 20 (only after step 3 looks right)

In [ ]:
sh(f"cd {REPO} && python scripts/fsl_pipeline.py --src {HCP} --dst {OUT}")
print(open(OUT + "/summary.csv").read())

## 5. Package results for download (do not publish them - HCP data terms)

In [ ]:
sh(f"cd /kaggle/working && rm -f hcp_processed/*/native_ras1mm.nii.gz hcp_processed/*/t2_mni_raw.nii.gz hcp_processed/*/fast_* && zip -qr hcp_processed.zip hcp_processed && ls -lh hcp_processed.zip")